In [1]:
import os
import sys
import pandas as pd
import json
import pickle

In [2]:
input_file = "join_est_record_joblightranges.txt"
output_file = "subquery_plans_joblightranges.json"

In [4]:
# get lines from input file
with open("joblightranges/" + input_file, "r") as f:
    lines = f.readlines()

dict_subqueries = {}

query_counter = -1
subquery_counter = 0

for line in lines:
    
    if line.find("query: 0") != -1:
        query_counter += 1
        subquery_counter = 0
        dict_subqueries[query_counter] = {}
        dict_subqueries[query_counter][subquery_counter] = {}
    elif line.find("query:") != -1:
        subquery_counter += 1
        dict_subqueries[query_counter][subquery_counter] = {}
        
    if line.find("inner_rel") != -1:
        inner_rel_ = True
    if line.find("outer_rel") != -1:
        inner_rel_ = False

    if line.find("RELOPTINFO") != -1:
        # get what is between parenthesis
        relname = line.split("(")[1].split(")")[0]
        
        if inner_rel_:
            dict_subqueries[query_counter][subquery_counter]["inner_rel"] = relname
        else:
            dict_subqueries[query_counter][subquery_counter]["outer_rel"] = relname
            
            

In [6]:
# dump in subquery_plans_jobjoin.json
with open(output_file, "w") as f:
    json.dump(dict_subqueries, f, indent=4)

In [7]:
# load subquery_plans_joblight.json
dict_subqueries = {}
with open(output_file) as f:
    dict_subqueries = json.load(f)

In [8]:
# read lines of /home/user/mayer/FactorJoin/imdb/jobjoin/job_join_queries.sql
with open("/home/user/mayer/FactorJoin/imdb/joblightranges/joblightranges_queries.sql", "r") as f:
    queries = f.readlines()

In [13]:
# read queries from files
def read_queries(queries, subqueries):
    dict_queries = {}
    dict_subqueries = {}
    query_counter = 0
    
    for q in queries:
        #q = q + ";"
        q_ = q.strip("\n")
        q_ = q_.strip(";")
        dict_queries[str(query_counter)] = q_
        
        subqueries_q = subqueries[str(query_counter)]
        
        dict_subqueries[str(query_counter)] = []
        # iteratate over values in subqueries_q
        for q in subqueries_q.values():
            outer_ = q['outer_rel']
            outer_ = outer_.split(" ")
            # sorrt outer_ to get the correct order
            outer_ = sorted(outer_)
            outer_ = " ".join(outer_)
            tuple_ = (q['inner_rel'], outer_)
            dict_subqueries[str(query_counter)].append(tuple_)
        
        query_counter += 1
    
    return (dict_queries, dict_subqueries)



In [14]:
dict_qs = read_queries(queries, dict_subqueries)


In [15]:
with open('dict_sql_imdb_joblightranges.pkl', 'wb') as f:
    pickle.dump(dict_qs, f)

In [16]:
dict_qs

({'0': "SELECT COUNT(*)  FROM movie_companies AS mc,  movie_info_idx AS mi_idx,  title AS t  WHERE t.id=mc.movie_id AND t.id=mi_idx.movie_id AND t.kind_id=2 AND t.series_years>='2004-2008' AND t.production_year<=2004.0",
  '1': "SELECT COUNT(*) FROM movie_companies AS mc, movie_info_idx AS mi_idx, title AS t WHERE t.id=mc.movie_id AND t.id=mi_idx.movie_id AND t.phonetic_code='S2532' AND mc.company_type_id=1 AND mi_idx.info_type_id=101 AND t.kind_id=1",
  '2': "SELECT COUNT(*) FROM movie_companies AS mc, movie_info_idx AS mi_idx, title AS t WHERE t.id=mc.movie_id AND t.id=mi_idx.movie_id AND t.kind_id=4 AND t.phonetic_code<='S3524' AND mc.company_type_id=1 AND t.production_year>=1987.0",
  '3': "SELECT COUNT(*) FROM movie_companies AS mc, movie_info_idx AS mi_idx, title AS t WHERE t.id=mc.movie_id AND t.id=mi_idx.movie_id AND t.production_year<=2012.0 AND t.season_nr=2.0 AND t.episode_nr>=8.0 AND t.phonetic_code>='N162' AND mi_idx.info_type_id=101 AND mc.company_type_id=1",
  '4': 'SELE